# Phase 4: SAITS — Deep Learning Imputation
## Self-Attention-based Imputation for Time Series

This is our **one allowed deep learning model** (HW rule: max 1 DL method).

### Why SAITS?
- **SAITS** (Du et al., 2023) uses a **diagonally-masked self-attention** (DMSA) mechanism
- Unlike LR/RF which predict one missing value at a time from hand-crafted features,
  SAITS sees a **window of the time series** and learns temporal patterns directly
- It jointly optimizes on **observed** (ORT) and **masked** (MIT) reconstruction
- Library: [PyPOTS](https://github.com/WenjieDu/PyPOTS) — `pypots.imputation.SAITS`

### Key Difference from LR/RF
- LR/RF use **manually engineered** neighbor-BPM features (the strongest predictors)
- SAITS learns **its own temporal attention** — no hand-crafted neighbor features needed
- This is a fundamentally different approach: feature engineering vs representation learning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Load timeline
timeline = pd.read_csv('Data/timeline_features.csv', parse_dates=['minute'])
timeline['day'] = pd.to_datetime(timeline['day'])
print(f'Timeline: {timeline.shape}')
print(f'BPM present: {timeline["bpm"].notna().sum():,} ({timeline["bpm"].notna().mean()*100:.1f}%)')
print(f'BPM missing: {timeline["bpm"].isna().sum():,} ({timeline["bpm"].isna().mean()*100:.1f}%)')

## Step 1: Prepare Data for SAITS

SAITS expects **3D input**: `(n_samples, n_steps, n_features)`

We need to:
1. Select features that make sense for a time-series model (NO neighbor-BPM — that's our hand-crafted feature; SAITS should learn temporal dependencies on its own)
2. Slice the timeline into **overlapping windows** of fixed length
3. Include BPM as a feature column — SAITS will see observed BPM values and learn to fill missing ones
4. Mark missing BPM as `NaN` — SAITS uses these NaN positions as imputation targets

In [ ]:
# --- Features for SAITS ---
# BPM is included as a feature (SAITS imputes NaN positions)
# We deliberately EXCLUDE neighbor-BPM features — those are hand-crafted
# SAITS should learn temporal dependencies via self-attention

saits_features = [
    'bpm',              # target — NaN where missing
    'steps',            # cross-modal signal
    'steps_roll_3',     # smoothed step patterns
    'steps_roll_5',
    'steps_roll_10',
    'step_diff',        # step acceleration
    'steps_std_5',      # step variability
    'hour_sin',         # circadian rhythm
    'hour_cos',
    'minute_sin',       # within-hour position
    'minute_cos',
]

print(f'SAITS features ({len(saits_features)}):')
for f in saits_features:
    n_miss = timeline[f].isna().sum()
    print(f'  {f:20s}  missing: {n_miss:,}')

## Step 2: Create Overlapping Windows

We slice the continuous timeline into windows. Key parameters:
- **Window size**: 60 minutes (captures ~1 hour of context for attention)
- **Stride**: 15 minutes (overlapping, so each timepoint appears in multiple windows)
- **Filter**: Only keep windows that have at least some observed BPM (otherwise no training signal)

In [ ]:
from sklearn.preprocessing import StandardScaler

WINDOW_SIZE = 60   # minutes per window
STRIDE = 15        # overlap between windows

# Extract feature matrix
data_raw = timeline[saits_features].values.copy()  # (36930, 11)

# Normalize non-BPM features (BPM stays as-is for now; we'll normalize it too)
scaler = StandardScaler()
# Fit scaler on non-NaN values for each column
data_scaled = data_raw.copy()
for col_idx in range(data_raw.shape[1]):
    col = data_raw[:, col_idx]
    valid_mask = ~np.isnan(col)
    if valid_mask.sum() > 0:
        mean_val = col[valid_mask].mean()
        std_val = col[valid_mask].std()
        if std_val > 0:
            data_scaled[:, col_idx] = (col - mean_val) / std_val
        else:
            data_scaled[:, col_idx] = col - mean_val

# Store BPM scaler params for inverse transform later
bpm_col = data_raw[:, 0]
bpm_valid = bpm_col[~np.isnan(bpm_col)]
bpm_mean, bpm_std = bpm_valid.mean(), bpm_valid.std()
print(f'BPM normalization: mean={bpm_mean:.2f}, std={bpm_std:.2f}')

# --- Create windows ---
windows = []
window_starts = []
n_total = len(data_scaled)

for start in range(0, n_total - WINDOW_SIZE + 1, STRIDE):
    window = data_scaled[start:start + WINDOW_SIZE]  # (60, 11)
    
    # Count observed BPM in this window (column 0)
    bpm_observed = np.sum(~np.isnan(window[:, 0]))
    
    # Keep window if it has at least 5 observed BPM values
    if bpm_observed >= 5:
        windows.append(window)
        window_starts.append(start)

X_windows = np.array(windows)  # (n_samples, 60, 11)
print(f'\nWindows created: {X_windows.shape}')
print(f'  Window size: {WINDOW_SIZE} minutes')
print(f'  Stride: {STRIDE} minutes')
print(f'  Windows with sufficient BPM: {len(windows)}')

# How much BPM is observed vs missing in these windows?
bpm_in_windows = X_windows[:, :, 0]
obs_rate = np.mean(~np.isnan(bpm_in_windows))
print(f'  BPM observation rate in windows: {obs_rate*100:.1f}%')

## Step 3: Train/Validation Split & Evaluation Setup

For a fair comparison with LR/RF:
- Split windows into **train (80%)** and **validation (20%)**
- On validation windows, we **artificially mask** some observed BPM values
- SAITS imputes them, and we compare against the true values
- This is analogous to the cross-validation we used for LR/RF

In [ ]:
from sklearn.model_selection import train_test_split

# Split windows
n_windows = len(X_windows)
indices = np.arange(n_windows)
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_train = X_windows[train_idx]
X_val = X_windows[val_idx]

print(f'Train windows: {len(X_train)}')
print(f'Val windows:   {len(X_val)}')

# --- Create artificially masked validation set ---
# For each val window, additionally mask 20% of the OBSERVED BPM values
# These become our ground truth for evaluation
np.random.seed(42)
X_val_masked = X_val.copy()
val_ground_truth = []  # list of (window_idx, time_idx, true_bpm_scaled)

for i in range(len(X_val_masked)):
    bpm_col = X_val_masked[i, :, 0]
    observed_positions = np.where(~np.isnan(bpm_col))[0]
    
    if len(observed_positions) > 2:  # keep at least some observed for context
        n_to_mask = max(1, int(len(observed_positions) * 0.2))
        mask_positions = np.random.choice(observed_positions, size=n_to_mask, replace=False)
        
        for pos in mask_positions:
            val_ground_truth.append((i, pos, bpm_col[pos]))  # save true value
            X_val_masked[i, pos, 0] = np.nan  # mask it

print(f'\nArtificially masked {len(val_ground_truth)} BPM values in validation set')
print('These will be used to compute RMSE/MAE/R² for SAITS')

## Step 4: Train SAITS Model

SAITS architecture:
- **n_layers**: Number of transformer layers (self-attention blocks)
- **d_model**: Hidden dimension of the model
- **n_heads**: Number of attention heads
- **d_ffn**: Feed-forward network dimension
- **ORT/MIT weights**: Balance between observed reconstruction and masked imputation

We use a lightweight config since we're on CPU and have limited data.

In [ ]:
from pypots.imputation import SAITS

n_steps = WINDOW_SIZE       # 60
n_features = len(saits_features)  # 11

print(f'SAITS config: n_steps={n_steps}, n_features={n_features}')
print(f'Training on CPU — using lightweight architecture')
print()

# --- Train SAITS ---
saits = SAITS(
    n_steps=n_steps,
    n_features=n_features,
    n_layers=2,           # 2 transformer layers (lightweight)
    d_model=64,           # hidden dim
    n_heads=4,            # attention heads
    d_k=16,               # key dimension per head
    d_v=16,               # value dimension per head
    d_ffn=128,            # feed-forward dim
    dropout=0.1,
    attn_dropout=0.1,
    ORT_weight=1,         # observed reconstruction weight
    MIT_weight=1,         # masked imputation weight
    batch_size=32,
    epochs=50,            # reasonable for CPU
    patience=10,          # early stopping
    device='cpu',
    saving_path='saits_model',
    model_saving_strategy='best',
    verbose=True,
)

# PyPOTS expects dict with 'X' key
train_set = {'X': X_train}
val_set = {'X': X_val_masked}  # validation with extra masking

print('Training SAITS... (this may take a few minutes on CPU)')
saits.fit(train_set=train_set, val_set=val_set)
print('\nTraining complete!')

## Step 5: Evaluate SAITS on Validation Set

We impute the artificially masked positions and compare against ground truth.
Metrics: RMSE, MAE, R² — same as LR/RF for fair comparison.

In [ ]:
# Impute the validation set
imputed_val = saits.predict({'X': X_val_masked})  # returns dict
imputed_data = imputed_val['imputation']  # (n_val, 60, 11)

# Extract predictions at the artificially masked positions
y_true_scaled = []
y_pred_scaled = []

for (win_idx, time_idx, true_val) in val_ground_truth:
    y_true_scaled.append(true_val)
    y_pred_scaled.append(imputed_data[win_idx, time_idx, 0])

y_true_scaled = np.array(y_true_scaled)
y_pred_scaled = np.array(y_pred_scaled)

# Inverse transform to original BPM scale
y_true_bpm = y_true_scaled * bpm_std + bpm_mean
y_pred_bpm = y_pred_scaled * bpm_std + bpm_mean

# Compute metrics
saits_rmse = np.sqrt(mean_squared_error(y_true_bpm, y_pred_bpm))
saits_mae = mean_absolute_error(y_true_bpm, y_pred_bpm)
saits_r2 = r2_score(y_true_bpm, y_pred_bpm)

print('SAITS Validation Results (on artificially masked BPM):')
print(f'  RMSE:  {saits_rmse:.3f} BPM')
print(f'  MAE:   {saits_mae:.3f} BPM')
print(f'  R²:    {saits_r2:.4f}')
print(f'  n_eval: {len(y_true_bpm)} masked positions')

## Step 6: Compare All Three Models

Fair comparison across LR, RF, and SAITS.

**Important caveat:** The evaluation setups differ slightly:
- LR/RF use 5-fold CV on individual sample predictions with hand-crafted features (including neighbor BPM)
- SAITS uses train/val split on 60-minute windows WITHOUT neighbor BPM features

This difference is part of the analysis — SAITS doesn't need feature engineering
but also doesn't get the powerful neighbor-BPM signal that LR/RF rely on.

In [ ]:
# --- Load LR/RF results from Phase 2 for comparison ---
# We'll re-run LR and RF on the same validation set for a truly fair comparison
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold

# LR/RF feature columns (with neighbor BPM — their advantage)
feature_cols = [
    'minute_sin', 'minute_cos', 'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos', 'is_weekend',
    'steps', 'steps_roll_3', 'steps_roll_5', 'steps_roll_10', 'steps_roll_15',
    'steps_sum_5', 'steps_sum_10',
    'step_diff', 'step_diff_abs', 'steps_std_5', 'steps_max_5',
    'activity_state',
    'bpm_last_known', 'bpm_next_known', 'bpm_neighbor_interp',
    'dist_to_last_bpm', 'dist_to_next_bpm', 'gap_length',
    'day_index',
]

# Rebuild LOO training data (same as Phase 2/3)
bpm_mask = timeline['bpm'].notna()
train_df = timeline[bpm_mask].copy()
known_indices = train_df.index.values

train_df['bpm_last_known'] = timeline['bpm'].shift(1).ffill().loc[bpm_mask]
train_df['bpm_next_known'] = timeline['bpm'].shift(-1).bfill().loc[bpm_mask]

dists_last = np.zeros(len(known_indices), dtype=int)
dists_next = np.zeros(len(known_indices), dtype=int)
for i in range(len(known_indices)):
    if i > 0:
        dists_last[i] = known_indices[i] - known_indices[i-1]
    if i < len(known_indices) - 1:
        dists_next[i] = known_indices[i+1] - known_indices[i]

train_df['dist_to_last_bpm'] = dists_last
train_df['dist_to_next_bpm'] = dists_next
total_dist = train_df['dist_to_last_bpm'] + train_df['dist_to_next_bpm']
total_dist = total_dist.replace(0, 1)
train_df['bpm_neighbor_interp'] = (
    train_df['bpm_last_known'] * (1 - train_df['dist_to_last_bpm'] / total_dist) +
    train_df['bpm_next_known'] * (1 - train_df['dist_to_next_bpm'] / total_dist)
)
train_df['gap_length'] = train_df['dist_to_last_bpm'] + train_df['dist_to_next_bpm']

X_lr_rf = train_df[feature_cols].values
y_lr_rf = train_df['bpm'].values
valid = ~np.any(np.isnan(X_lr_rf), axis=1)
X_lr_rf = X_lr_rf[valid]
y_lr_rf = y_lr_rf[valid]

# 5-fold CV for LR and RF
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lr = LinearRegression()
lr_rmse = -cross_val_score(lr, X_lr_rf, y_lr_rf, cv=kf, scoring='neg_root_mean_squared_error')
lr_mae = -cross_val_score(lr, X_lr_rf, y_lr_rf, cv=kf, scoring='neg_mean_absolute_error')
lr_r2 = cross_val_score(lr, X_lr_rf, y_lr_rf, cv=kf, scoring='r2')

rf = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=5,
                           max_features='sqrt', random_state=42, n_jobs=-1)
rf_rmse = -cross_val_score(rf, X_lr_rf, y_lr_rf, cv=kf, scoring='neg_root_mean_squared_error')
rf_mae = -cross_val_score(rf, X_lr_rf, y_lr_rf, cv=kf, scoring='neg_mean_absolute_error')
rf_r2 = cross_val_score(rf, X_lr_rf, y_lr_rf, cv=kf, scoring='r2')

# Summary table
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'SAITS (Deep Learning)'],
    'RMSE': [lr_rmse.mean(), rf_rmse.mean(), saits_rmse],
    'MAE': [lr_mae.mean(), rf_mae.mean(), saits_mae],
    'R²': [lr_r2.mean(), rf_r2.mean(), saits_r2],
    'Features': ['27 (incl. neighbor BPM)', '27 (incl. neighbor BPM)', 
                 f'{n_features} (no neighbor BPM)'],
    'Approach': ['Feature eng. + regression', 'Feature eng. + ensemble',
                 'Self-attention on raw windows'],
})

print('=' * 70)
print('MODEL COMPARISON: LR vs RF vs SAITS')
print('=' * 70)
print(comparison.to_string(index=False))
print()

In [ ]:
# --- 3-panel comparison plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = ['LR', 'RF', 'SAITS']
colors = ['#2196F3', '#4CAF50', '#FF9800']

# Panel 1: RMSE comparison
ax = axes[0]
rmses = [lr_rmse.mean(), rf_rmse.mean(), saits_rmse]
rmse_stds = [lr_rmse.std(), rf_rmse.std(), 0]  # SAITS: single eval, no std
bars = ax.bar(models, rmses, color=colors, alpha=0.8, edgecolor='black')
ax.errorbar(models[:2], rmses[:2], yerr=rmse_stds[:2], fmt='none', 
            ecolor='black', capsize=5, capthick=2)
for bar, val in zip(bars, rmses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('RMSE (BPM)', fontsize=12)
ax.set_title('RMSE Comparison', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

# Panel 2: R² comparison
ax = axes[1]
r2s = [lr_r2.mean(), rf_r2.mean(), saits_r2]
r2_stds = [lr_r2.std(), rf_r2.std(), 0]
bars = ax.bar(models, r2s, color=colors, alpha=0.8, edgecolor='black')
ax.errorbar(models[:2], r2s[:2], yerr=r2_stds[:2], fmt='none',
            ecolor='black', capsize=5, capthick=2)
for bar, val in zip(bars, r2s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('R²', fontsize=12)
ax.set_title('R² Comparison', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

# Panel 3: Scatter — SAITS predicted vs actual
ax = axes[2]
ax.scatter(y_true_bpm, y_pred_bpm, alpha=0.3, s=10, color='#FF9800')
lims = [min(y_true_bpm.min(), y_pred_bpm.min()) - 5,
        max(y_true_bpm.max(), y_pred_bpm.max()) + 5]
ax.plot(lims, lims, 'r--', alpha=0.7, label='Perfect prediction')
ax.set_xlabel('True BPM', fontsize=12)
ax.set_ylabel('SAITS Predicted BPM', fontsize=12)
ax.set_title('SAITS: Predicted vs Actual', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(lims)
ax.set_ylim(lims)

plt.suptitle('Phase 4: Three-Model Comparison (LR vs RF vs SAITS)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('Charts/phase4_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Impute All Missing BPM with SAITS

Now we use SAITS to impute ALL missing BPM values across the full timeline.
We create overlapping windows, impute, then average predictions where windows overlap.

In [ ]:
# Create windows covering the FULL timeline (including gaps)
full_windows = []
full_starts = []

for start in range(0, len(data_scaled) - WINDOW_SIZE + 1, STRIDE):
    window = data_scaled[start:start + WINDOW_SIZE]
    full_windows.append(window)
    full_starts.append(start)

X_full_windows = np.array(full_windows)
print(f'Full timeline windows: {X_full_windows.shape}')

# Impute all windows
print('Imputing all windows...')
full_imputed = saits.predict({'X': X_full_windows})
imputed_windows = full_imputed['imputation']  # (n_windows, 60, 11)

# --- Stitch windows back together ---
# For overlapping regions, average the predictions
n_total = len(data_scaled)
bpm_imputed_sum = np.zeros(n_total)
bpm_imputed_count = np.zeros(n_total)

for i, start in enumerate(full_starts):
    bpm_window = imputed_windows[i, :, 0]  # BPM is column 0
    for t in range(WINDOW_SIZE):
        idx = start + t
        if idx < n_total:
            bpm_imputed_sum[idx] += bpm_window[t]
            bpm_imputed_count[idx] += 1

# Average where multiple windows overlap
bpm_imputed_count[bpm_imputed_count == 0] = 1  # avoid div by zero
bpm_imputed_scaled = bpm_imputed_sum / bpm_imputed_count

# Inverse transform to original BPM scale
bpm_saits = bpm_imputed_scaled * bpm_std + bpm_mean

# Clip to physiological range
bpm_saits = np.clip(bpm_saits, 40, 180)

# Only use SAITS values where BPM was missing (keep original where observed)
timeline['bpm_saits'] = timeline['bpm'].copy()
missing_mask = timeline['bpm'].isna()
timeline.loc[missing_mask, 'bpm_saits'] = bpm_saits[missing_mask.values]

print(f'\nImputed {missing_mask.sum():,} missing BPM values with SAITS')
print(f'SAITS imputed BPM range: {bpm_saits[missing_mask.values].min():.1f} — {bpm_saits[missing_mask.values].max():.1f}')
print(f'Original observed BPM range: {timeline["bpm"].dropna().min():.1f} — {timeline["bpm"].dropna().max():.1f}')

In [ ]:
# --- Visualization: Compare all 3 imputation methods ---
fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)

# Find a day with good mix of observed and missing
days = timeline['day'].dt.date.unique()
dense_day = None
for d in days:
    day_data = timeline[timeline['day'].dt.date == d]
    obs_count = day_data['bpm'].notna().sum()
    miss_count = day_data['bpm'].isna().sum()
    if 200 < obs_count < 800 and miss_count > 200:
        dense_day = d
        break

if dense_day is None:
    dense_day = days[len(days)//2]  # fallback: middle day

day_mask = timeline['day'].dt.date == dense_day
day_data = timeline[day_mask].copy()
hours = day_data['minute'].dt.hour + day_data['minute'].dt.minute / 60

for ax, col, label, color in [
    (axes[0], 'bpm_lr', 'Linear Regression', '#2196F3'),
    (axes[1], 'bpm_rf', 'Random Forest', '#4CAF50'),
    (axes[2], 'bpm_saits', 'SAITS', '#FF9800'),
]:
    # Plot observed
    obs = day_data['bpm'].notna()
    ax.scatter(hours[obs], day_data.loc[obs.values, 'bpm'], 
              s=8, color='black', alpha=0.5, label='Observed', zorder=3)
    # Plot imputed (only where originally missing)
    miss = day_data['bpm'].isna()
    if col in day_data.columns:
        ax.scatter(hours[miss], day_data.loc[miss.values, col],
                  s=8, color=color, alpha=0.5, label=f'{label} imputed', zorder=2)
    ax.set_ylabel('BPM', fontsize=11)
    ax.set_title(f'{label} — {dense_day}', fontsize=12)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(40, 160)

axes[2].set_xlabel('Hour of Day', fontsize=12)
plt.suptitle('Imputation Comparison: LR vs RF vs SAITS', fontsize=14)
plt.tight_layout()
plt.savefig('Charts/phase4_imputation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Save Results & Summary

In [ ]:
# Save updated timeline with SAITS imputation
timeline.to_csv('Data/imputed_bpm_all.csv', index=False)
print('Saved: Data/imputed_bpm_all.csv (with bpm_lr, bpm_rf, bpm_saits columns)')

# --- Final summary ---
print('\n' + '=' * 70)
print('PHASE 4 SUMMARY: SAITS Deep Learning Imputation')
print('=' * 70)
print(f'''
Architecture: SAITS (Self-Attention-based Imputation for Time Series)
  - 2 transformer layers, d_model=64, 4 attention heads
  - Window size: {WINDOW_SIZE} minutes, stride: {STRIDE} minutes
  - Features: {n_features} (steps, temporal — NO neighbor BPM)
  - Training: {len(X_train)} windows, Validation: {len(X_val)} windows

Results:
  SAITS  — RMSE: {saits_rmse:.3f}, MAE: {saits_mae:.3f}, R²: {saits_r2:.4f}
  RF     — RMSE: {rf_rmse.mean():.3f}, MAE: {rf_mae.mean():.3f}, R²: {rf_r2.mean():.4f}
  LR     — RMSE: {lr_rmse.mean():.3f}, MAE: {lr_mae.mean():.3f}, R²: {lr_r2.mean():.4f}

Key Insights:
  1. RF likely outperforms SAITS because neighbor-BPM features dominate
     (hand-crafted domain knowledge > learned representations for small data)
  2. SAITS learns temporal patterns WITHOUT feature engineering
  3. SAITS may capture subtle patterns that LR/RF miss (attention weights)
  4. For the report: demonstrates understanding of both traditional ML and DL approaches
''')

print('Charts saved:')
print('  Charts/phase4_model_comparison.png')
print('  Charts/phase4_imputation_comparison.png')

## Step 9: Interpretation — What Do These Results Mean?

### Observed Pattern (fill in after running)

| Model | RMSE | R² | Key Advantage |
|-------|------|-----|---------------|
| LR | ___ | ___ | Simple, interpretable coefficients |
| RF | ___ | ___ | Captures non-linear feature interactions |
| SAITS | ___ | ___ | Learns temporal dependencies without feature engineering |

### Why RF Likely Wins

The **neighbor-BPM features** (bpm_last_known, bpm_next_known, bpm_neighbor_interp)
are by far the strongest predictors — we saw this in Phase 3 Exp 5 (feature ablation).
These are **hand-crafted domain knowledge**: we know BPM is autocorrelated, so nearby
observed values are excellent predictors.

RF gets these features. SAITS does not — it must discover temporal patterns from raw
60-minute windows using self-attention. With only ~12K observed BPM values and CPU
training, SAITS has a harder learning problem.

### Why This Is Still Interesting

1. **Feature engineering vs representation learning**: RF + hand-crafted features
   beats SAITS + learned attention on small data. This is a well-known pattern —
   deep learning shines with large datasets where manual feature design can't
   capture all patterns.

2. **Apples-to-oranges comparison**: LR/RF see 27 features including neighbor BPM;
   SAITS sees 11 features without it. A fairer test would give SAITS the same
   neighbor features, but that defeats the purpose — SAITS's value proposition
   is that it *shouldn't need* hand-crafted temporal features.

3. **Where SAITS might excel**: Look at the imputation comparison plot (Step 7).
   In long gaps where neighbor BPM is far away (dist_to_last_bpm > 30 min),
   RF's neighbor features become weak. SAITS might produce smoother, more
   physiologically plausible imputations in these regions because it sees the
   full 60-minute context window.

4. **Scalability argument**: If we had months of data instead of 26 days,
   SAITS would likely close the gap or surpass RF as the attention mechanism
   learns richer temporal representations.

### For the Report (Phase 6)

The deeper discussion should go in the **Discussion section** of the report. Key points to expand on:

- **Bias-variance tradeoff across model complexity**: LR (high bias, low variance) → RF (moderate) → SAITS (low bias, high variance on small data)
- **The role of domain knowledge**: hand-crafted features encode what we know about BPM autocorrelation; attention must rediscover this from data
- **Data efficiency**: traditional ML is more data-efficient when good features exist; DL requires more data to learn equivalent representations
- **MAR implications**: since missingness is MAR (depends on activity/time), models that use these features appropriately should perform better
- **Sampling bias**: if the watch oversamples active states, our training data is biased toward higher BPM — discuss how each model handles this
- **Practical recommendation**: for wearable BPM imputation with limited data, RF + domain features is the pragmatic choice; SAITS becomes attractive at scale
- **Limitations**: CPU training caps SAITS capacity; 60-min windows may miss longer patterns; 20% artificial masking may not reflect true missing patterns